### TypedDict vs dataclass vs Pydantic — three ways to shape data

Before building LangGraph state machines, it helps to know there are three different tools for "giving data a shape" in Python — and LangGraph specifically picks one of them for a reason.

**Analogy: a relay race.**
- **`TypedDict`** is the baton — passed hand to hand at speed between runners (nodes). Nobody stops to check what's inside it; it's just a label so *you* remember what it's supposed to hold. Python enforces nothing at runtime.
- **`dataclass`** is your personal training log — structured, easy to fill in, but nobody else audits it. It's for your own private bookkeeping, never handed to an outsider.
- **Pydantic's `BaseModel`** is passport control at the airport — every value is checked before it's allowed through, because it's crossing a boundary where you don't yet trust what's arriving (a user's request, an LLM's tool-call output, another service's response).

This notebook shows what each one actually does at runtime — including what happens when the data is *wrong* — then a decision guide for which to reach for.

#### 1. `TypedDict` — the relay baton (this is what LangGraph state uses)

LangGraph's `StateGraph` passes state between node functions as a plain `dict`. `TypedDict` gives you autocomplete and type hints in your editor for that dict's shape — but Python does **not** check any of it while the program is running.

In [1]:
from typing import TypedDict

class PortfolioState(TypedDict):
    amount_usd: float
    total_usd: float
    total_inr: float

# TypedDict is just a hint for your editor -- Python does not check this at runtime.
bad_state: PortfolioState = {"amount_usd": "not a number", "total_usd": 0, "total_inr": 0}
print(bad_state)  # runs fine, no error, even though amount_usd should be a float

{'amount_usd': 'not a number', 'total_usd': 0, 'total_inr': 0}


#### 2. `dataclass` — your private training log

A `dataclass` gives you a real class with attributes, a free `__init__`, and a free `__repr__` — useful for internal objects you mutate as your program runs. Like `TypedDict`, it does **not** validate the values you put in.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class JobState:
    status: str = "running"
    logs: list[str] = field(default_factory=list)

job = JobState(status=123)  # status should be a str -- dataclass does not care
print(job)

#### 3. Pydantic's `BaseModel` — passport control

Pydantic validates the data the moment you construct the object and raises immediately if the shape is wrong. As a bonus, it can generate a JSON schema for the model automatically — exactly what FastAPI request bodies and MCP tool schemas need.

In [ ]:
from pydantic import BaseModel, ValidationError

class PortfolioRequest(BaseModel):
    amount_usd: float
    total_usd: float
    total_inr: float

try:
    PortfolioRequest(amount_usd="not a number", total_usd=0, total_inr=0)
except ValidationError as exc:
    print(exc)  # rejected immediately, with a precise error pointing at amount_usd

In [ ]:
# Bonus: Pydantic can hand you a JSON schema for free.
# This is what FastAPI uses for OpenAPI docs, and what MCP servers use to
# describe a tool's expected input to an LLM agent.
import json
print(json.dumps(PortfolioRequest.model_json_schema(), indent=2))

#### Side by side

| | `TypedDict` | `dataclass` | Pydantic `BaseModel` |
|---|---|---|---|
| Runtime validation | No | No | Yes — raises on bad data |
| Generates JSON schema | No | No | Yes, automatically |
| Mutation cost | Cheapest (plain dict) | Cheap (plain object) | Slower (validates on every construction) |
| Crosses a trust boundary? | No — internal graph state only | No — internal only | Yes — this is the whole point |

#### When to use what

- **Building a LangGraph `StateGraph`?** Use `TypedDict`. That's literally what `StateGraph(YourState)` expects, and node functions run many times per invocation — you want zero validation overhead on every single node call.
- **Tracking mutable internal state that never leaves your process** (a background job tracker, an in-memory cache entry, a counter you update as you go)? Use a `dataclass`. You get a real object with defaults and a readable `__repr__`, with no validation tax and no need for a wire format.
- **Data is arriving from outside your code** — a user's HTTP request body, an LLM's tool-call arguments, another service's API response? Use Pydantic's `BaseModel`. You need it validated, and you need the JSON schema it generates for free (FastAPI's OpenAPI docs, an MCP tool's input schema).

**One-line rule of thumb:** does this data cross a boundary where you don't already trust it?
- Yes → Pydantic.
- No, but a framework expects it shaped like a plain mapping → `TypedDict`.
- No, and it's just your own bookkeeping → `dataclass`.